# Computational Experiments for Minimum Embedding Dimension GNS

We consider generalized numerical semigroups

$$
S=S(T,\{d_2\})
\subseteq \mathbb N^2,
$$

where

$$
T=\langle a,b\rangle $$

is a numerical semigroup generated by two coprime integers.

The goal is to experimentally study the genus

$$ g(S) $$

using the description of gaps provided by Theorem 3.1.

In [7]:
from math import gcd
import pandas as pd

## Numerical Semigroup Utilities

For

$$ T=\langle a,b\rangle, $$ 

we compute:

- membership in $T$,
- the gap set $H(T)$,
- the Frobenius number.

In [9]:
def in_T(n, a, b):
    """
    Membership test for T=<a,b>.
    """
    for i in range(n // a + 1):
        if (n - i * a) % b == 0:
            return True
    return False


def frobenius(a, b):
    return a * b - a - b


def gaps_T(a, b):
    """
    Gap set H(T).
    """
    F = frobenius(a, b)

    return [
        n
        for n in range(F + 1)
        if not in_T(n, a, b)
    ]

## Computing α(h)

For a gap $h$, the quantity

$$ \alpha(h) $$

is the length of the consecutive gap interval ending at $h$.

In [10]:
def alpha_values(a, b):

    gaps = gaps_T(a, b)
    gap_set = set(gaps)

    alpha = {}

    for h in gaps:

        start = h

        while start - 1 in gap_set:
            start -= 1

        alpha[h] = h - start + 1

    return alpha

## Gap Intervals

We decompose the gap set into maximal consecutive intervals

$$ I_1,\ldots,I_r.$$ 

The lengths of these intervals are useful for describing
the genus of the associated generalized numerical semigroup.

In [11]:
def gap_intervals(a, b):

    gaps = gaps_T(a, b)

    if not gaps:
        return []

    intervals = []

    start = gaps[0]
    prev = gaps[0]

    for g in gaps[1:]:

        if g == prev + 1:
            prev = g
        else:
            intervals.append((start, prev))
            start = g
            prev = g

    intervals.append((start, prev))

    return intervals


def interval_lengths(a, b):

    return [
        end - start + 1
        for start, end in gap_intervals(a, b)
    ]

## Genus of the Generalized Numerical Semigroup

For dimension two, Theorem 3.1 implies

$$ g(S) = d_2 \sum_{h\in H(T)} \alpha(h).$$ 

We compute this quantity directly.

In [12]:
def genus_S(a, b, d2):

    alpha = alpha_values(a, b)

    return d2 * sum(alpha.values())

## Example

In [13]:
a = 4
b = 7
d2 = 3

print("Gaps:", gaps_T(a, b))
print("Alpha values:", alpha_values(a, b))
print("Gap interval lengths:", interval_lengths(a, b))
print("Genus g(S):", genus_S(a, b, d2))

Gaps: [1, 2, 3, 5, 6, 9, 10, 13, 17]
Alpha values: {1: 1, 2: 2, 3: 3, 5: 1, 6: 2, 9: 1, 10: 2, 13: 1, 17: 1}
Gap interval lengths: [3, 2, 2, 1, 1]
Genus g(S): 42


## Large Scale Experiment

We generate all coprime pairs

$$ 2 \le a < b \le 50 $$


and all

$$   1 \le d_2 \le 10.$$


For each example we record:

- generators $a,b$,
- Frobenius number,
- genus of $T$,
- interval lengths,
- genus of $S$.

In [21]:
def predicted_genus(a, b, d2):

    lengths = interval_lengths(a, b)

    return d2 * sum(
        L * (L + 1) // 2
        for L in lengths
    )

Do we have the following?

$$ \sum_{h\in H(T)} \alpha(h) = \sum_i \frac{\alpha_i(\alpha_i + 1)}{2}$$
If this is true then one has
$$ g(S) = d_2 \sum_i \frac{\alpha_i(\alpha_i + 1)}{2}$$

In [22]:
rows = []

for a in range(2, 51):

    for b in range(a + 1, 51):

        if gcd(a, b) != 1:
            continue

        gaps = gaps_T(a, b)

        genus_T = len(gaps)

        lengths = interval_lengths(a, b)

        for d2 in range(1, 11):

            rows.append(
                {
                    "a": a,
                    "b": b,
                    "d2": d2,
                    "frobenius": frobenius(a, b),
                    "genus_T": genus_T,
                    "num_gap_intervals": len(lengths),
                    "gap_interval_lengths": str(lengths),
                    "genus_S": genus_S(a, b, d2),
                    "predicted_genus":
                        predicted_genus(a, b, d2),

                    "difference":
                        genus_S(a, b, d2)
                        - predicted_genus(a, b, d2),
                }
            )

df = pd.DataFrame(rows)

df.head()

,a,b,d2,frobenius,genus_T,num_gap_intervals,gap_interval_lengths,genus_S,predicted_genus,difference
0,2,3,1,1,1,1,[1],1,1,0
1,2,3,2,1,1,1,[1],2,2,0
2,2,3,3,1,1,1,[1],3,3,0
3,2,3,4,1,1,1,[1],4,4,0
4,2,3,5,1,1,1,[1],5,5,0


## Save Results

In [23]:
bad_rows = df[df["difference"] != 0]

print("Number of failures:", len(bad_rows))
bad_rows.head()

Number of failures: 0


,a,b,d2,frobenius,genus_T,num_gap_intervals,gap_interval_lengths,genus_S,predicted_genus,difference


In [24]:
csv_file = "minimum_embedding_dimension_gns.csv"

df.to_csv(csv_file, index=False)

print(f"Saved {len(df)} rows to {csv_file}")

Saved 7240 rows to minimum_embedding_dimension_gns.csv


In [25]:
df

,a,b,d2,frobenius,genus_T,num_gap_intervals,gap_interval_lengths,genus_S,predicted_genus,difference
0,2,3,1,1,1,1,[1],1,1,0
1,2,3,2,1,1,1,[1],2,2,0
2,2,3,3,1,1,1,[1],3,3,0
3,2,3,4,1,1,1,[1],4,4,0
4,2,3,5,1,1,1,[1],5,5,0
...,...,...,...,...,...,...,...,...,...,...
7235,49,50,6,2351,1176,48,"[48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 3...",117600,117600,0
7236,49,50,7,2351,1176,48,"[48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 3...",137200,137200,0
7237,49,50,8,2351,1176,48,"[48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 3...",156800,156800,0
7238,49,50,9,2351,1176,48,"[48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 3...",176400,176400,0


## It seems that the predicted formula for the genus is probably correct. 